# The Retrieval Problem — companion notebook

This notebook is the **narrative** half of the topic's Python pillar. The reusable,
tested implementation lives next door in [`the_retrieval_problem.py`](the_retrieval_problem.py);
here we import it and walk the topic section by section, so every claim the page makes
renders as executed output. The `.py` *owns the numbers* — its `assert`-based harness
encodes each pedagogical claim, and running it as a script is the regression test.

Run the reference end-to-end:

```bash
uv run --with numpy python notebooks/the-retrieval-problem/the_retrieval_problem.py
```


In [ ]:
import sys
import pathlib

# Make the_retrieval_problem.py importable whether the kernel starts in the repo
# root or in this notebook's own directory.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "the-retrieval-problem", pathlib.Path("notebooks/the-retrieval-problem")):
    if (_cand / "the_retrieval_problem.py").exists():
        sys.path.insert(0, str(_cand))
        break

import numpy as np
import the_retrieval_problem as trp


## 1. Retrieval as ranking by a relevance functional

Retrieval scores every document by a relevance functional and returns the top few.
Here the functional is one of three geometric similarity scores between the query
direction `q` and each document vector. We read off the dot product, cosine, and
Euclidean distance for the finance corpus — note that the document **norms vary by
design**, which is what makes the three scores disagree below.


In [ ]:
q = trp._Q
print(f"query q = {q.tolist()}  (unit direction 'interest rate exposure')\n")
print(f"{'document':<16}{'vector':>16}{'norm':>9}{'dot':>9}{'cosine':>9}{'euclid':>9}")
for name, v in trp._CORPUS.items():
    print(f"{name:<16}{str(v.tolist()):>16}{np.linalg.norm(v):>9.4f}"
          f"{trp.dot(q, v):>9.4f}{trp.cosine(q, v):>9.4f}{trp.euclidean(q, v):>9.4f}")


## 2. Three similarity scores and one identity

The three scores are tied together by a single identity,
$\lVert a-b\rVert^2 = \lVert a\rVert^2 + \lVert b\rVert^2 - 2\langle a, b\rangle$,
which the harness checks across dimensions up to 64.


In [ ]:
trp.test_cosine_distance_identity()

## 3. On the unit sphere the three rankings coincide

Once every vector is L2-normalized onto the unit sphere, the cosine, dot-product,
and (ascending) Euclidean rankings induce the **same order** — the keystone result.
The harness verifies it on the finance corpus and on random corpora up to 128
dimensions.


In [ ]:
trp.test_on_sphere_rankings_coincide()

sphere = {n: trp.normalize(v) for n, v in trp._CORPUS.items()}
print("on-sphere ranking:", trp.rank(trp.normalize(q), sphere, trp.cosine, descending=True))


## 4. Off the sphere they diverge: magnitude matters

On the raw (unnormalized) corpus the dot product rewards the large-norm padded
transcript, while cosine — which quotients out magnitude — keeps the concise
on-point filing at the top. Scaling a document's norm reorders the dot ranking but
never the cosine ranking. This is the magnitude sensitivity that BM25's length
normalization later removes.


In [ ]:
trp.test_off_sphere_dot_cosine_disagree()
trp.test_magnitude_scaling_flips_dot()
trp.test_cosine_scale_invariance_and_dot_linearity()


## 5. Which of these are metrics?

Euclidean distance satisfies all four metric axioms. Cosine "distance" $1-\cos$ does
not — it violates the triangle inequality, which the harness demonstrates with an
explicit three-point counterexample and reports the violation gap. The dot product
is not a distance at all (no identity of indiscernibles, and a vector need not be
its own best match).


In [ ]:
trp.test_euclidean_is_metric()
gap = trp.test_cosine_distance_violates_triangle()
print(f"triangle-inequality violation gap = {gap:.4f}")
trp.test_dot_product_not_a_metric()


## 6. Level sets: the decision geometry

The locus of points with equal score to the query is a **hyperplane** for the dot
product, a **sphere** for Euclidean distance, and a **cone** for cosine. The harness
samples each locus and confirms the score is constant along it.


In [ ]:
trp.test_level_sets()

## 7. Finance case study

The full table the topic and the interactive laboratory mirror to the decimal: the
dot ranking led by the padded transcript, the cosine ranking led by the on-point
filing, their coincidence on the sphere, and the magnitude threshold at which the
dot top-1 flips.


In [ ]:
trp.finance_demo()
trp.test_finance_demo_numbers()


---

Every number above is asserted in [`the_retrieval_problem.py`](the_retrieval_problem.py):
the cosine–distance identity, monotone rank-invariance, the on-sphere equivalence, the
off-sphere divergence and its flip threshold, the metric axioms for Euclidean distance,
the cosine-distance triangle violation, and the hyperplane/sphere/cone level sets. The
three pillars — the proofs on the page, the interactive laboratory, and this tested code
— agree by construction.
